# Using library Requests
---
**Author**: Marko Bajec

**Last update**: 3.3.2026

**Description**: the library <code>Requests</code> is a high-level HTTP library for Python. It can be easily used for fetching, as it support many useful features, such as *Keep-Alive & Connection Pooling*, *Sessions with Cookie Persistence*, *Proxy Handling*, *Connection Timeouts*, etc. 

This notebook shows few examples of using <code>Request</code> for fetching pages. 

**Official web page:** http://docs.python-requests.org/. For more details check [here](http://www.python-requests.org/en/latest/api/#classes).

---

In [1]:
import requests
# url samples
url1 = 'http://github.com'
url2 = 'http://www.times.si'
url3 = 'http://www.delo.si'
url4 = 'http://dev.vitabits.org'
url5 = 'https://en.knu.ac.kr/main/main.htm'
url6 = 'http://fri.uni-lj.si'

### Simple fetch using http GET request

In [2]:
url = url2
response = requests.get(url)
print('status code:', response.status_code)
print('url:', response.url)
print('content:', response.text[:1000])

status code: 200
url: https://www.times.si/
content: <!doctype html><html lang="sl"><head><!-- Google tag (gtag.js) --><script async src="https://www.googletagmanager.com/gtag/js?id=G-VLKSG5FWDE"></script><script>window.dataLayer = window.dataLayer || [];function gtag(){dataLayer.push(arguments);}gtag('js', new Date()); gtag('config', 'G-VLKSG5FWDE');</script><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1, shrink-to-fit=no"><link rel="stylesheet" href="https://stackpath.bootstrapcdn.com/bootstrap/4.3.1/css/bootstrap.min.css" integrity="sha384-ggOyR0iXCbMQv3Xipma34MD+dH/1fQ784/j6cY/iJTQUOhcWr7x9JvoRxT2MZw1T" crossorigin="anonymous"><link rel=stylesheet href="/s/css/times.css"><title>Vse novice na enem mestu - TIMES.si</title><meta name="keywords" content="novice,sveže novice,zadnje novice,slovenija,šport,gospodarstvo,svet,evropa,smrt,nesreča,tehnologija,24ur,rtvslo" /><meta name="description" content="Stran zbira, kategorizira in združuje sorodn

### What is requests returning?

In [ ]:
url = url6
response = requests.get(url, allow_redirects=True)
print('status code:', response.status_code)
print('url:', response.url)
print('content:', response.text)

### Identifying redirections

In [ ]:
url = url3
response = requests.get(url, verify=True, allow_redirects=True, timeout=50)
print('status code:', response.status_code)
print('starting url:', url3)
print('ending url:', response.url)
print('history:', response.history)
print('headers:', response.headers)
# note the attribute history. If not empty, it tells what had happened before we got to the last URL, 
# in our case https://www.delo.si/. Remember that we called unsecure http//www.delo.si and not https://www.delo.si

### Blocking redirections

In [ ]:
url = url3
response = requests.get(url, verify=False, allow_redirects=False, timeout=50)
print('status code:', response.status_code)
print('url:', response.url)
print('history:', response.history)
print('headers:', response.headers)

### http HEAD request

In [8]:
url = url2
response = requests.head(url)
print('status code:', response.status_code)
print('url:', response.url)
print('headers:', response.headers)
print('text:', response.text)
# note that since we made http HEAD request, the response.text attribute is empty. 

status code: 301
url: http://www.times.si/
headers: {'Server': 'nginx/1.14.0 (Ubuntu)', 'Date': 'Tue, 03 Mar 2026 08:42:24 GMT', 'Content-Type': 'text/html', 'Content-Length': '194', 'Connection': 'keep-alive', 'Location': 'https://www.times.si/'}
text: 


### Handling exceptions
**Error and exception handling** is of utter importance for crawlers that need to be robust in order to visit a large portion of the web. <code>Requests</code> lib can catch several types of exceptions. 

Use <code>raise_for_status()</code> 

In [ ]:
url = url5
try:
    response = requests.get(url, timeout=5)
    print("final url:", response.url)
    print("status:", response.status_code)
    print("server:", response.headers.get("Server"))
    print("content-type:", response.headers.get("Content-Type"))
    print("first 300 chars of body:\n", response.text[:300])
    response.raise_for_status()
except requests.HTTPError:
    print('An HTTP error occurred.')
except requests.ConnectionError:
    print('A Connection error occurred.')
except requests.URLRequired:
    print('A valid URL is required to make a request.')
except requests.TooManyRedirects:
    print('Too many redirects.')
except requests.ConnectTimeout:
    print('The request timed out while trying to connect to the remote server.')
except requests.ReadTimeout:
    print('The server did not send any data in the allotted amount of time.')
except requests.Timeout:
    print('The request timed out.')
except requests.RequestException as e:
    print(e)
except:
    print('Unknown error occured!')
    raise

An optimized version of error handling. Less code, same coverage: <code>RequestException</code> already covers <code>ConnectionError</code>, <code>URLRequired</code>, <code>TooManyRedirects</code>, <code>Timeout</code>, etc.

In [ ]:
from requests.exceptions import RequestException

def fetch(url, timeout=5, max_body=300):
    try:
        r = requests.get(url, timeout=timeout)
        print("final url:", r.url)
        print("status:", r.status_code)
        print("server:", r.headers.get("Server"))
        print("content-type:", r.headers.get("Content-Type"))
        print(f"first {max_body} chars of body:\n", r.text[:max_body])
        r.raise_for_status()
        return r
    except RequestException as e:
        print("Request error:", e)
        if getattr(e, "response", None) is not None:
            print("status:", e.response.status_code)
        return None
    
url = url5

fetch(url)

### Sending browser-like header

Sometimes a web server my reject your request if the client is not recognized as a browser

In [ ]:
url = url5

headers = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) "
                   "Chrome/122.0.0.0 Safari/537.36"),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Connection": "keep-alive",
}

with requests.Session() as s:
    r = s.get(url, headers=headers, timeout=10)
    print(r.status_code, r.url)
    print(r.text[:200])
    r.raise_for_status()

### Basic authentication
<code>HTTP Basic Auth</code> used to be very common authentication mechanism for web services. <code>Requests</code> supports it straight out of the box. More common today is however <code>OAuth2.0</code> authentication mechanism. Check [here](https://oauth.net/) for more information.

#### Simplest way

In [ ]:
url = "https://httpbin.org/basic-auth/user/passwd"  # demo endpoint
r = requests.get(url, auth=("user", "passwd"), timeout=10)
print(r.status_code)
print(r.json())

#### Using HTTPBasicAuth explicitly
Useful if you prefer being explicit or want to swap auth types easily.

In [14]:
from requests.auth import HTTPBasicAuth

url = "https://httpbin.org/basic-auth/user/passwd"
r = requests.get(url, auth=HTTPBasicAuth("user", "passwd"), timeout=10)

print(r.status_code)
print(r.json())

200
{'authenticated': True, 'user': 'user'}


#### Reuse Auth across many requests (Session)

In [15]:
from requests.auth import HTTPBasicAuth

session = requests.Session()
session.auth = HTTPBasicAuth("user", "passwd")

r1 = session.get("https://httpbin.org/basic-auth/user/passwd", timeout=10)
print("r1:", r1.status_code, r1.json())

# Any further requests use the same Basic Auth automatically:
r2 = session.get("https://httpbin.org/anything", timeout=10)
print("r2:", r2.status_code)

r1: 200 {'authenticated': True, 'user': 'user'}
r2: 200


#### What’s actually sent (Authorization header)
Basic Auth is just an Authorization header with base64(user:pass).

In [16]:
import base64

user, pwd = "user", "passwd"
token = base64.b64encode(f"{user}:{pwd}".encode()).decode()
headers = {"Authorization": f"Basic {token}"}

url = "https://httpbin.org/basic-auth/user/passwd"
r = requests.get(url, headers=headers, timeout=10)
print(r.status_code, r.json())

200 {'authenticated': True, 'user': 'user'}
